In [8]:
import os
import random
import numpy as np
import pandas as pd
import sys
from itertools import product

from sklearn.metrics import (
    accuracy_score, f1_score, recall_score, precision_score, confusion_matrix
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# 실행 환경 확인(재현성/버전 이슈 점검용)
print("TensorFlow:", tf.__version__)
print("Python:", sys.executable)

TensorFlow: 2.21.0
Python: c:\Users\cj020\OneDrive\Labtop\KDISS\TS_RL_proj\venv\Scripts\python.exe


In [9]:
# ADASYN 및 스케일링이 이미 반영된 분할 데이터 로드
train_df = pd.read_csv(r"../../data/processed/ADASYN/data_train_adasyn.csv")
valid_df = pd.read_csv(r"../../data/processed/ADASYN/data_valid.csv")
test_df = pd.read_csv(r"../../data/processed/ADASYN/data_test.csv")

print("train/valid/test shape:")
print("train:", train_df.shape)
print("valid:", valid_df.shape)
print("test :", test_df.shape)

print("\ncolumns:")
print(train_df.columns.tolist())

# 타깃 컬럼명
target_col = "Risk_Label"

print("\ntrain y distribution:")
print(train_df[target_col].astype(int).value_counts())
print(train_df[target_col].astype(int).value_counts(normalize=True))

train/valid/test shape:
train: (2351, 10)
valid: (1438, 10)
test : (822, 10)

columns:
['GJR_VaR_5_t1', 'NASDAQ_return(%)', 'VKOSPI_Close', 'Brent Crude Oil_return(%)', 'Gold Spot_return(%)', 'KOSDAQ_return(%)', 'KOSPI 200 lagged_1_return(%)', 'KOSPI 200_BB_width', 'KOSPI 200_OG', 'Risk_Label']

train y distribution:
Risk_Label
0    1638
1     713
Name: count, dtype: int64
Risk_Label
0    0.696725
1    0.303275
Name: proportion, dtype: float64


In [10]:
# -------------------------
# 0. 시드 고정(실험 재현성 확보)
# -------------------------
SEED = 1

def set_seed(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(SEED)

# -------------------------
# 1. 입력/타깃 분리
# -------------------------
X_train_raw = train_df.drop(columns=[target_col]).copy()
y_train = train_df[target_col].astype(int).copy()

X_valid_raw = valid_df.drop(columns=[target_col]).copy()
y_valid = valid_df[target_col].astype(int).copy()

X_test_raw = test_df.drop(columns=[target_col]).copy()
y_test = test_df[target_col].astype(int).copy()

# 전부 수치형인지 확인
non_numeric_cols = X_train_raw.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric_cols:
    raise ValueError(
        f"ANN 입력에는 수치형만 넣는 게 안전합니다. 비수치형 컬럼: {non_numeric_cols}"
    )

# valid/test 컬럼을 train 컬럼 순서와 일치시켜 추론 오류 방지
X_valid_raw = X_valid_raw[X_train_raw.columns]
X_test_raw = X_test_raw[X_train_raw.columns]

# -------------------------
# 2. 입력 데이터는 이미 스케일링된 상태
# -------------------------
X_train_scaled = X_train_raw.copy()
X_valid_scaled = X_valid_raw.copy()
X_test_scaled = X_test_raw.copy()

# -------------------------
# 3. Keras 입력용 ndarray 변환
# -------------------------
X_train_np = np.asarray(X_train_scaled, dtype=np.float32)
y_train_np = np.asarray(y_train, dtype=np.float32)

X_valid_np = np.asarray(X_valid_scaled, dtype=np.float32)
y_valid_np = np.asarray(y_valid, dtype=np.float32)

X_test_np = np.asarray(X_test_scaled, dtype=np.float32)
y_test_np = np.asarray(y_test, dtype=np.float32)

print("train shape:", X_train_np.shape, y_train_np.shape)
print("train class 비율:")
print(pd.Series(y_train_np.astype(int)).value_counts(normalize=True).sort_index())


train shape: (2351, 9) (2351,)
train class 비율:
0    0.696725
1    0.303275
Name: proportion, dtype: float64


In [11]:
# 단일 은닉층 ANN 생성 함수
from pathlib import Path


def build_ann(input_dim, hidden_units=41, activation="relu"):
    return Sequential([
        Input(shape=(input_dim,)),
        Dense(hidden_units, activation=activation),
        Dense(1, activation="sigmoid")
    ])

# 평가 함수: cutoff 선택과 최종 평가에서 공통 사용
# G-Mean 중심으로 선택/평가.
def compute_binary_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).reshape(-1)
    y_pred = (y_prob >= threshold).astype(int)

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    prec = precision_score(y_true, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    gmean = float(np.sqrt(rec * specificity))

    return {
        "accuracy": acc,
        "f1": f1,
        "recall": rec,
        "precision": prec,
        "specificity": specificity,
        "gmean": gmean,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "y_pred": y_pred,
        "y_prob": y_prob
    }

# 탐색 대상: 소스 논문 기준 설정(hidden_units=41, ReLU, lr=5.8e-4)을 포함하고,
# 기본 구조는 단일 은닉층 ANN + Adam + early stopping으로 유지
hidden_units_grid = [32, 41, 64, 128]
activation_grid = ["relu", "tanh"]
lr_grid = [1e-3, 5.8e-4, 5e-4, 1e-4]
optimizer_grid = ["adam"]

# 모든 조합 생성
search_space = [
    {
        "hidden_units": hidden_units,
        "activation": activation,
        "lr": lr,
        "optimizer": optimizer
    }
    for hidden_units, activation, lr, optimizer in product(
        hidden_units_grid,
        activation_grid,
        lr_grid,
        optimizer_grid
    )
]

# 선택 우선순위: valid_gmean > valid_recall > valid_f1 > valid_precision > valid_log_loss
# cutoff는 0.5 고정이 아니라 valid set에서 함께 선택
search_history = []

for i, cfg in enumerate(search_space, start=1):
    tf.keras.backend.clear_session()
    set_seed(SEED)

    model = build_ann(
        input_dim=X_train_np.shape[1],
        hidden_units=cfg["hidden_units"],
        activation=cfg["activation"]
    )

    if cfg["optimizer"] == "adam":
        optimizer = Adam(learning_rate=cfg["lr"])
    else:
        raise ValueError(f"지원하지 않는 optimizer: {cfg['optimizer']}")

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    # 학습 중단은 val_loss 기준(과적합 완화)
    es = EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=20,
        restore_best_weights=True,
        verbose=0
    )

    history = model.fit(
        X_train_np,
        y_train_np,
        validation_data=(X_valid_np, y_valid_np),
        epochs=300,
        batch_size=32,
        callbacks=[es],
        verbose=0
    )

    valid_prob = model.predict(X_valid_np, verbose=0).ravel()
    valid_loss = float(np.min(history.history["val_loss"]))

    # 모델별 확률 분포에 맞춰 cutoff 후보 생성 + 0.5도 후보에 포함
    threshold_grid = np.unique(
        np.r_[
            np.unique(valid_prob),
            0.5,
            valid_prob.min() - 1e-12,
            valid_prob.max() + 1e-12
        ]
    )
    threshold_grid = threshold_grid[(threshold_grid >= 0) & (threshold_grid <= 1)]

    cutoff_rows = []
    for cutoff in threshold_grid:
        m = compute_binary_metrics(y_valid_np, valid_prob, threshold=float(cutoff))
        cutoff_rows.append({
            "threshold": float(cutoff),
            "accuracy": m["accuracy"],
            "f1": m["f1"],
            "recall": m["recall"],
            "precision": m["precision"],
            "specificity": m["specificity"],
            "gmean": m["gmean"]
        })

    cutoff_df = (
        pd.DataFrame(cutoff_rows)
        .sort_values(
            by=["gmean", "recall", "f1", "precision"],
            ascending=[False, False, False, False]
        )
        .reset_index(drop=True)
    )
    best_cutoff_row = cutoff_df.iloc[0]

    row = {
        **cfg,
        "best_cutoff": float(best_cutoff_row["threshold"]),
        "valid_accuracy": float(best_cutoff_row["accuracy"]),
        "valid_f1": float(best_cutoff_row["f1"]),
        "valid_recall": float(best_cutoff_row["recall"]),
        "valid_precision": float(best_cutoff_row["precision"]),
        "valid_specificity": float(best_cutoff_row["specificity"]),
        "valid_gmean": float(best_cutoff_row["gmean"]),
        "valid_log_loss": valid_loss,
        "epochs_run": len(history.history["loss"])
    }
    search_history.append(row)

    print(
        f"[Config {i:02d}] {cfg} -> "
        f"cutoff={row['best_cutoff']:.6f}, "
        f"valid_gmean={row['valid_gmean']:.6f}, "
        f"valid_recall={row['valid_recall']:.6f}, "
        f"valid_f1={row['valid_f1']:.6f}, "
        f"valid_precision={row['valid_precision']:.6f}, "
        f"best_val_loss={row['valid_log_loss']:.6f}"
    )

search_df = (
    pd.DataFrame(search_history)
    .sort_values(
        by=["valid_gmean"],
        ascending=[False]
    )
    .reset_index(drop=True)
)

# G-Mean 상위 1%만 선택
top_1_percent_gmean = search_df.head(int(len(search_df) * 0.01) or 1)

# 상위 1% 중 Precision이 가장 높은 값 선택
best_row = (
    top_1_percent_gmean
    .sort_values(
        by=["valid_precision", "valid_gmean", "valid_recall", "valid_f1", "valid_log_loss"],
        ascending=[False, False, False, False, True]
    )
    .iloc[0]
)

best_config = {
    "hidden_units": int(best_row["hidden_units"]),
    "activation": str(best_row["activation"]),
    "lr": float(best_row["lr"]),
    "optimizer": str(best_row["optimizer"])
}
best_cutoff = float(best_row["best_cutoff"])

print("\n[Top 10 ANN GMean Search Results]")
print(search_df.head(10).to_string(index=False))

print("\n선택된 설정:", best_config)
print(f"선택된 cutoff: {best_cutoff:.6f}")
print(
    f"선택 기준 valid_gmean={best_row['valid_gmean']:.6f}, "
    f"valid_recall={best_row['valid_recall']:.6f}, "
    f"valid_f1={best_row['valid_f1']:.6f}, "
    f"valid_precision={best_row['valid_precision']:.6f}, "
    f"best_val_loss={best_row['valid_log_loss']:.6f}"
)

# 선택된 설정으로 최종 모델 재학습
tf.keras.backend.clear_session()
set_seed(SEED)

ann_model = build_ann(
    input_dim=X_train_np.shape[1],
    hidden_units=best_config["hidden_units"],
    activation=best_config["activation"]
)
ann_model.compile(
    optimizer=Adam(learning_rate=best_config["lr"]),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=20,
    restore_best_weights=True,
    verbose=1
)

history = ann_model.fit(
    X_train_np,
    y_train_np,
    validation_data=(X_valid_np, y_valid_np),
    epochs=300,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

[Config 01] {'hidden_units': 32, 'activation': 'relu', 'lr': 0.001, 'optimizer': 'adam'} -> cutoff=0.289785, valid_gmean=0.674526, valid_recall=0.653846, valid_f1=0.348463, valid_precision=0.237525, best_val_loss=0.423991
[Config 02] {'hidden_units': 32, 'activation': 'relu', 'lr': 0.00058, 'optimizer': 'adam'} -> cutoff=0.307847, valid_gmean=0.671810, valid_recall=0.626374, valid_f1=0.352396, valid_precision=0.245161, best_val_loss=0.432316
[Config 03] {'hidden_units': 32, 'activation': 'relu', 'lr': 0.0005, 'optimizer': 'adam'} -> cutoff=0.309703, valid_gmean=0.667747, valid_recall=0.620879, valid_f1=0.348228, valid_precision=0.241970, best_val_loss=0.433198
[Config 04] {'hidden_units': 32, 'activation': 'relu', 'lr': 0.0001, 'optimizer': 'adam'} -> cutoff=0.296281, valid_gmean=0.667714, valid_recall=0.626374, valid_f1=0.346505, valid_precision=0.239496, best_val_loss=0.424141
[Config 05] {'hidden_units': 32, 'activation': 'tanh', 'lr': 0.001, 'optimizer': 'adam'} -> cutoff=0.290938,

In [14]:
# =========================
# 마지막 셀. ANN GMean 예측 결과 저장
# =========================
from pathlib import Path
import pandas as pd
import numpy as np

result_dir = Path("../../results/results_ML")
result_dir.mkdir(parents=True, exist_ok=True)

# 날짜 불러오기
date_df = pd.read_csv("../../data/processed/data_selected.csv")
date_df["Date"] = pd.to_datetime(date_df["Date"])
date_df = date_df.sort_values("Date").reset_index(drop=True)

# 예측값 생성
valid_prob = ann_model.predict(X_valid_np, verbose=0).ravel()
test_prob = ann_model.predict(X_test_np, verbose=0).ravel()

valid_pred = (valid_prob >= best_cutoff).astype(int)
test_pred = (test_prob >= best_cutoff).astype(int)

# 날짜 매칭
n_valid = len(valid_pred)
n_test = len(test_pred)

valid_dates = date_df["Date"].tail(n_valid + n_test).head(n_valid).dt.strftime("%Y-%m-%d").to_numpy()
test_dates = date_df["Date"].tail(n_test).dt.strftime("%Y-%m-%d").to_numpy()

# 저장
pd.DataFrame({
    "Date": valid_dates,
    "ANN_valid_pred_gmean": valid_pred
}).to_csv(result_dir / "03. ANN_valid_pred.csv", index=False, encoding="utf-8-sig")

pd.DataFrame({
    "Date": test_dates,
    "ANN_test_pred_gmean": test_pred
}).to_csv(result_dir / "03. ANN_test_pred.csv", index=False, encoding="utf-8-sig")

print("저장 완료")
print(result_dir / "03. ANN_valid_pred_gmean.csv")
print(result_dir / "03. ANN_test_pred_gmean.csv")

저장 완료
..\..\results\results_ML\03. ANN_valid_pred_gmean.csv
..\..\results\results_ML\03. ANN_test_pred_gmean.csv


In [13]:
print("\nValid Confusion Matrix:")
print(confusion_matrix(y_valid_np, valid_pred, labels=[0, 1]))

print("\nTest Confusion Matrix:")
print(confusion_matrix(y_test_np, test_pred, labels=[0, 1]))


Valid Confusion Matrix:
[[907 349]
 [ 66 116]]

Test Confusion Matrix:
[[434 289]
 [ 34  65]]
